# WolBanking77 : classification d'intentions bancaires en wolof

Ce notebook est volontairement simple. Il compare une baseline Machine Learning et AfroXLMR-114L, un Transformer adapte aux langues africaines et incluant le wolof. Hugging Face Trainer gere l'entrainement, la validation et les checkpoints.

## 1. Importer le projet depuis GitHub et monter Drive

GitHub contient le code et les CSV. Google Drive conserve les modeles et les rapports apres la fermeture de Colab.

In [ ]:
from google.colab import drive
from pathlib import Path

GITHUB_REPOSITORY_URL = 'https://github.com/VOTRE_COMPTE/WolBanking77.git'
PROJECT_DIR = Path('/content/WolBanking77')
DRIVE_DIR = Path('/content/drive/MyDrive/WolBanking77_runs')

drive.mount('/content/drive')
if not PROJECT_DIR.exists():
    !git clone $GITHUB_REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR

assert (PROJECT_DIR / 'data').exists(), 'Le depot GitHub doit contenir le dossier data.'
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print('Projet :', PROJECT_DIR)
print('Sauvegardes :', DRIVE_DIR)

## 2. Installer les bibliotheques et configurer

Activez un GPU Colab avec Runtime > Change runtime type > T4 GPU. Commencez avec 5k_split, puis utilisez full lorsque tout fonctionne.

In [ ]:
!pip -q install 'transformers>=4.44,<5.0' datasets accelerate sentencepiece joblib

import json
import joblib
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments

DATASET_NAME = '5k_split'
MODEL_NAME = 'Davlan/afro-xlmr-base-114L'
TEXT_COLUMN, LABEL_COLUMN = 'input_wo', 'label'
MAX_LENGTH, BATCH_SIZE, EPOCHS = 128, 16, 10
DATA_DIR = PROJECT_DIR / 'data' / DATASET_NAME
MODEL_DIR = DRIVE_DIR / f'afroxlmr_{DATASET_NAME}'
REPORT_DIR = DRIVE_DIR / 'reports'
MODEL_DIR.mkdir(exist_ok=True); REPORT_DIR.mkdir(exist_ok=True)
print('GPU disponible :', torch.cuda.is_available())

## 3. Fonction load_data

Charge les CSV, nettoie les textes vides ou dupliques et encode les 77 intentions.

In [ ]:
def load_data():
    train = pd.read_csv(DATA_DIR / 'train' / 'train.csv')
    test = pd.read_csv(DATA_DIR / 'test' / 'test.csv')
    for frame in (train, test):
        frame.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN], inplace=True)
        frame['text'] = frame[TEXT_COLUMN].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
        frame.drop(frame.index[frame['text'].eq('')], inplace=True)
        frame.drop_duplicates(subset=['text'], inplace=True)
    encoder = LabelEncoder()
    train['label_id'] = encoder.fit_transform(train[LABEL_COLUMN])
    test['label_id'] = encoder.transform(test[LABEL_COLUMN])
    return train.reset_index(drop=True), test.reset_index(drop=True), encoder

train_df, test_df, label_encoder = load_data()
print(f'Train={len(train_df)}, Test={len(test_df)}, Intentions={len(label_encoder.classes_)}')
display(train_df[['input_wo', 'label']].head())

## 4. Baseline simple : TF-IDF + LinearSVC

Cette baseline rapide donne une reference concrete. Le Transformer doit la depasser sur le macro-F1.

In [ ]:
baseline = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 5), sublinear_tf=True)),
    ('classifier', LinearSVC(C=1.0)),
])
baseline.fit(train_df['text'], train_df['label'])
baseline_pred = baseline.predict(test_df['text'])
baseline_metrics = {'model': 'TF-IDF + LinearSVC', 'accuracy': accuracy_score(test_df['label'], baseline_pred), 'macro_f1': f1_score(test_df['label'], baseline_pred, average='macro', zero_division=0)}
joblib.dump(baseline, DRIVE_DIR / f'baseline_{DATASET_NAME}.joblib')
print(baseline_metrics)

## 5. Fonction tokenize

Separe 10 % du train pour la validation. Le test officiel reste reserve a la mesure finale.

In [ ]:
train_part, validation_part = train_test_split(train_df, test_size=0.10, random_state=42, stratify=train_df['label_id'])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

def make_dataset(frame):
    dataset = Dataset.from_pandas(frame[['text', 'label_id']], preserve_index=False).rename_column('label_id', 'labels')
    return dataset.map(tokenize, batched=True, remove_columns=['text'])

train_ds = make_dataset(train_part)
validation_ds = make_dataset(validation_part)
test_ds = make_dataset(test_df)
print('Train / validation / test :', len(train_ds), len(validation_ds), len(test_ds))

## 6. Fonction compute_metrics

Le macro-F1 est la mesure principale, car chaque intention compte autant.

In [ ]:
def compute_metrics(prediction):
    logits, labels = prediction
    predicted = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, predicted), 'macro_f1': f1_score(labels, predicted, average='macro', zero_division=0)}

## 7. Entrainer AfroXLMR-114L

Trainer ecrit un checkpoint dans Drive a chaque epoque et conserve automatiquement le meilleur modele selon le macro-F1 de validation. L'arret anticipe limite le surapprentissage.

In [ ]:
id2label = {index: label for index, label in enumerate(label_encoder.classes_)}
label2id = {label: index for index, label in id2label.items()}
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label_encoder.classes_), id2label=id2label, label2id=label2id)

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR), learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS, weight_decay=0.01,
    evaluation_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
    save_total_limit=2, fp16=torch.cuda.is_available(), report_to='none', seed=42,
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=validation_ds,
    tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()
trainer.save_model(MODEL_DIR / 'best_model')
tokenizer.save_pretrained(MODEL_DIR / 'best_model')
with open(MODEL_DIR / 'labels.json', 'w', encoding='utf-8') as file:
    json.dump(label_encoder.classes_.tolist(), file, ensure_ascii=False, indent=2)
print('Meilleur modele :', MODEL_DIR / 'best_model')

## 8. Evaluation finale et comparaison

Les rapports sont sauvegardes dans Drive. Le Transformer evalue ici le test officiel pour la premiere fois.

In [ ]:
result = trainer.predict(test_ds)
transformer_pred = label_encoder.inverse_transform(np.argmax(result.predictions, axis=-1))
transformer_metrics = {'model': 'AfroXLMR-114L', 'accuracy': accuracy_score(test_df['label'], transformer_pred), 'macro_f1': f1_score(test_df['label'], transformer_pred, average='macro', zero_division=0)}

comparison = pd.DataFrame([baseline_metrics, transformer_metrics]).set_index('model')
display(comparison.style.format('{:.2%}'))
report = pd.DataFrame(classification_report(test_df['label'], transformer_pred, output_dict=True, zero_division=0)).T
report.to_csv(REPORT_DIR / f'afroxlmr_report_{DATASET_NAME}.csv')
with open(REPORT_DIR / f'comparison_{DATASET_NAME}.json', 'w', encoding='utf-8') as file:
    json.dump({'baseline': baseline_metrics, 'transformer': transformer_metrics}, file, ensure_ascii=False, indent=2)

## 9. Fonction predict_intent

Classifie une nouvelle requete avec le meilleur checkpoint garde par Trainer.

In [ ]:
def predict_intent(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LENGTH).to(trainer.model.device)
    with torch.no_grad():
        probabilities = torch.softmax(trainer.model(**inputs).logits, dim=1)[0]
    index = int(probabilities.argmax())
    return {'intention': label_encoder.inverse_transform([index])[0], 'confiance': round(float(probabilities[index]), 4)}

predict_intent('Dama bëgg xam xaalis bi nekk ci sama kont.')